In [ ]:
using Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames, Interpolations
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("plot_classification.jl")
include("plot_power_energy.jl")
include("utils.jl")

In [ ]:
# Modifying backend GR attributes
default(fmt = :png);

# Power separation computation

In [ ]:
mean_P = zeros(50)
mean_P_BMRU = zeros(50)
mean_P_other = zeros(50)

for i = 45 : 94

    # Full power extraction
    Ipower = 0
    time = 0
    time, Ipower = get_bistable_transient_data("./Cadence_output/$(i)_pow.vcsv")
    P = Ipower .* 1.8 * 1e9

    fig, total_energy = plot_power_energy(P, time; save_path="./Cadence_plots/power_energy_$(i).pdf")

    # Splitted power extraction
    time_BMRU, Ipower_BMRU = get_bistable_transient_data("./Cadence_output/$(i)_P_BMRU.vcsv")
    time_other, Ipower_other = get_bistable_transient_data("./Cadence_output/$(i)_P_other.vcsv")
    P_BMRU = Ipower_BMRU .* 1.8 * 1e9
    P_other = Ipower_other .* 1.8 * 1e9
    
    fig_BMRU, total_energy_BMRU = plot_power_energy(P_BMRU, time_BMRU; save_path="./Cadence_plots/power_energy_$(i)_BMRU.pdf")
    fig_other, total_energy_other = plot_power_energy(P_other, time_other; save_path="./Cadence_plots/power_energy_$(i)_other.pdf")

    # Saving data
    mean_P[i-44] = total_energy
    mean_P_BMRU[i-44] = total_energy_BMRU
    mean_P_other[i-44] = total_energy_other
end

In [ ]:
x = vcat(fill(1, length(mean_P)),
         fill(2, length(mean_P_BMRU)),
         fill(3, length(mean_P_other)))
values = vcat(mean_P, mean_P_BMRU, mean_P_other)

fig_power = violin(x, values;
       ylabel = L"\mathrm{Power}\,\,\mathrm{(nW)}",
       linewidth = 0,
       fillalpha = 0.75,
       legend = false,
       palette = :seaborn_muted,
       title = "",
       ylims = (0, 125),
       xticks = ([1, 2, 3], [L"\mathrm{Total}", L"\mathrm{BMRU}", L"\mathrm{FC}"]))

boxplot!(x, values;
         fillalpha = 0.3,
         linewidth = 1.5,
         markersize = 2,
         legend = false)

savefig(fig_power, "./Cadence_plots/power_separation.pdf")